# LIGO GW150914: an hour of gravitational-wave strain

GWOSC publishes 4096-second strain files at 4096 Hz and 16384 Hz.
That is 16.8 million or 67.1 million samples in a single line. At the
full-hour overview GW150914 is invisible in the noise; zoom around
`t = 0` and XY refines the decimated line until the chirp appears.

The default 4 kHz HDF5 file is about 134 MB. Set
`LIGO_SAMPLE_RATE=16384` for the full-rate, roughly 536 MB file.

**Source:** [GWOSC GW150914 event page](https://gwosc.org/events/GW150914/)
and [GWOSC URL lookup documentation](https://gwosc.readthedocs.io/en/stable/locate.html).
GWOSC event data are released under CC BY 4.0; follow the
acknowledgement guidance linked from the event page.

Install beside XY with
`python -m pip install numpy requests h5py gwosc xy`.


In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import requests
from gwosc.locate import get_event_urls

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "gwosc"
DATA_DIR.mkdir(parents=True, exist_ok=True)

EVENT = "GW150914"
EVENT_GPS = 1_126_259_462.4
DETECTOR = os.getenv("LIGO_DETECTOR", "H1")
SAMPLE_RATE = int(os.getenv("LIGO_SAMPLE_RATE", "4096"))
DURATION = 4096
if SAMPLE_RATE not in {4096, 16384}:
    raise ValueError("LIGO_SAMPLE_RATE must be 4096 or 16384")

urls = get_event_urls(
    EVENT,
    catalog="GWTC-1-confident",
    version=3,
    detector=DETECTOR,
    duration=DURATION,
    sample_rate=SAMPLE_RATE,
    format="hdf5",
)
if not urls:
    raise RuntimeError("GWOSC returned no matching strain file")
url = urls[0]
hdf5_path = DATA_DIR / Path(url).name
if not hdf5_path.exists():
    with requests.get(
        url,
        stream=True,
        timeout=(30, 3600),
    ) as response:
        response.raise_for_status()
        partial = hdf5_path.with_suffix(".hdf5.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                output.write(chunk)
        partial.replace(hdf5_path)

print(f"cached strain file: {hdf5_path}")

In [ ]:
with h5py.File(hdf5_path, "r") as data:
    strain = np.asarray(data["strain"]["Strain"], dtype=np.float64)
    gps_start = float(np.asarray(data["meta"]["GPSstart"]))
    x_spacing = float(
        data["strain"]["Strain"].attrs.get(
            "Xspacing",
            1 / SAMPLE_RATE,
        )
    )

seconds_from_event = gps_start + np.arange(strain.size, dtype=np.float64) * x_spacing - EVENT_GPS
print(
    f"{strain.size:,} samples · {1 / x_spacing:,.0f} Hz · "
    f"{strain.nbytes / 2**20:,.1f} MiB canonical strain"
)

In [ ]:
full_x_bounds = (
    float(seconds_from_event[0]),
    float(seconds_from_event[-1]),
)
full_y_bounds = (
    float(np.nanmin(strain)),
    float(np.nanmax(strain)),
)

chart = xy.line_chart(
    xy.line(
        seconds_from_event,
        strain,
        name=f"{DETECTOR} · raw strain",
        color="#43f4ff",
        width=1.35,
        opacity=0.92,
    ),
    xy.x_band(
        -0.18,
        0.02,
        text="EXPECTED SIGNAL WINDOW",
        color="#ff3f81",
        opacity=0.07,
        style={"font_size": 11, "font_weight": 700, "letter_spacing": "0.08em"},
    ),
    xy.hline(0, color="#2b7680", width=1, opacity=0.75),
    xy.vline(
        0,
        text="t = 0  ·  COALESCENCE",
        color="#ff4f91",
        width=2.25,
        opacity=0.95,
        style={
            "dash": "7,5",
            "font_size": 11,
            "font_weight": 700,
            "letter_spacing": "0.06em",
        },
    ),
    xy.text(
        0,
        4.45e-19,
        "COALESCENCE  /  t = 0",
        dx=-10,
        dy=-8,
        color="#ff4f91",
        anchor="end",
        style={"font_size": 11, "font_weight": 700, "letter_spacing": "0.06em"},
    ),
    xy.x_axis(
        label="TIME FROM COALESCENCE  /  seconds",
        domain=(-0.20, 0.08),
        bounds=full_x_bounds,
        tick_values=[-0.20, -0.15, -0.10, -0.05, 0, 0.05],
        tick_labels=["-0.20", "-0.15", "-0.10", "-0.05", "0", "+0.05"],
        style={
            "grid_color": "#173640",
            "grid_width": 1,
            "grid_dash": "dotted",
            "grid_opacity": 0.9,
            "axis_color": "#3d7380",
            "axis_width": 1,
            "tick_color": "#43f4ff",
            "tick_width": 1,
            "tick_length": 5,
            "tick_label_color": "#a9ccd2",
            "label_color": "#d7f9fc",
            "tick_label_size": 11,
            "label_size": 12,
        },
    ),
    xy.y_axis(
        label="RAW DETECTOR STRAIN  h(t)",
        label_offset=-18,
        domain=(-5e-19, 5e-19),
        bounds=full_y_bounds,
        tick_values=[-4e-19, -2e-19, 0, 2e-19, 4e-19],
        tick_labels=["-4e-19", "-2e-19", "0", "2e-19", "4e-19"],
        style={
            "grid_color": "#173640",
            "grid_width": 1,
            "grid_dash": "dotted",
            "grid_opacity": 0.9,
            "axis_color": "#3d7380",
            "axis_width": 1,
            "tick_color": "#43f4ff",
            "tick_width": 1,
            "tick_length": 5,
            "tick_label_color": "#a9ccd2",
            "label_color": "#d7f9fc",
            "tick_label_size": 11,
            "label_size": 12,
        },
    ),
    xy.tooltip(
        title=f"{DETECTOR} raw strain",
        format={"x": "+.5f", "y": ".3e"},
    ),
    xy.legend(show=False),
    xy.interaction_config(
        hover=True,
        crosshair=True,
        wheel_zoom=True,
        box_zoom=True,
        double_click_reset=True,
    ),
    xy.theme(
        background="#04080b",
        plot_background="#061116",
        text_color="#d7f9fc",
        grid_color="#173640",
        axis_color="#3d7380",
        crosshair_color="#ff4f91",
        selection_color="#43f4ff",
        selection_fill="#43f4ff22",
    ),
    title=(
        f"GW150914 SIGNAL LAB  ·  {DETECTOR} RAW STRAIN  ·  "
        f"{1 / x_spacing:,.0f} Hz  ·  {strain.size / 1e6:.1f}M SAMPLES"
    ),
    styles={
        "title": {
            "text_align": "left",
            "font_size": 18,
            "font_weight": 700,
            "letter_spacing": "0.06em",
        },
        "tick_label": {"font_family": "ui-monospace, monospace"},
        "axis_title": {
            "font_family": "ui-monospace, monospace",
            "letter_spacing": "0.05em",
        },
        "annotation_label": {"font_family": "ui-monospace, monospace"},
        "tooltip": {
            "background": "#07151a",
            "color": "#d7f9fc",
            "border": "1px solid #2b7680",
            "border_radius": 4,
            "font_family": "ui-monospace, monospace",
        },
    },
    style={"border": "1px solid #173640", "font_family": "ui-monospace, monospace"},
    width=1150,
    height=620,
    padding=(72, 32, 80, 156),
)
payload = chart.figure().build_payload()[0]
print("render tier:", payload["traces"][0]["tier"])
chart